## What is Multi-Agent Workflow?

A structured flow where multiple agents collaborate over multiple steps with control logic

## Key Componenets 

## What is different 

| Feature         | Delegation | Workflow |
| --------------- | ---------- | -------- |
| State tracking  | ❌          | ✅        |
| Error handling  | ❌          | ✅        |
| Dynamic routing | ❌          | ✅        |
| Extensible      | ❌          | ✅        |


In [5]:
import requests

AGENTS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]

def discover_capabilities():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()

        registry.append({
            "agent": info["name"],
            "description": info["description"],
            "endpoint": info["endpoint"],
            "tools": tools["tools"]
        })

    return registry

In [9]:
import re

def plan(query):
    query = query.lower()
    numbers = list(map(float, re.findall(r"\d+\.?\d*", query)))

    steps = []

    if "interest" in query:
        steps.append({
            "type": "finance",
            "tool": "calculate_interest",
            "args": {
                "principal": numbers[0],
                "rate": numbers[1],
                "time": numbers[2]
            }
        })

    if "add" in query:
        steps.append({
            "type": "math",
            "tool": "add",
            "args": {
                "b": numbers[-1]
            }
        })

    return steps

In [10]:
def build_payload(step, state):
    args = step["args"].copy()

    if step["tool"] == "add":
        args["a"] = state["result"]

    return {
        "tool": step["tool"],
        "args": args
    }

In [11]:
def execute(query):
    registry = discover_capabilities()
    steps = plan(query)

    state = {"result": None}

    for step in steps:

        # ROUTING
        agent = next(
            a for a in registry
            if step["tool"] in [t["name"] for t in a["tools"]]
        )

        # PAYLOAD
        payload = build_payload(step, state)

        # INVOCATION
        response = requests.post(agent["endpoint"], json=payload).json()

        if response.get("status") == "success":
            state["result"] = response["result"]
        else:
            print("⚠️ Step failed")

    return state["result"]

In [12]:
query = "calculate interest for 2000 at 10 for 1 year and then add 500"

print("\n🎯 FINAL:", execute(query))


🎯 FINAL: 700.0
